In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [22]:
df=pd.read_csv('feature_engineered_dataset.csv')

In [23]:
df.head(2)

,DATE,WEEK,MONTH,GYM TYPE,BODYWEIGHT,PHASE,EXERCISE,MUSCLE GROUP,WEIGHT,REPS,VOLUME,ESTIMATED 1RM,RELATIVE STRENGTH
0,2025-04-07,15,Apr,College Gym,70.85,Bulk,Squats,Legs,27.5,12,330.0,38.5,0.543402
1,2025-04-07,15,Apr,College Gym,70.85,Bulk,Squats,Legs,30.0,9,270.0,39.0,0.550459


In [24]:
df.shape

(2398, 13)

In [25]:
weekly_df = (
    df
    .groupby(
        ["WEEK", "MONTH", "MUSCLE GROUP", "GYM TYPE", "PHASE"],
        as_index=False
    )
    .agg(
        AVG_E1RM=("ESTIMATED 1RM", "mean"),
        AVG_RELATIVE_STRENGTH=("RELATIVE STRENGTH", "mean"),
        TOTAL_VOLUME=("VOLUME", "sum")
    )
)


In [26]:
weekly_df = weekly_df.sort_values(
    ["MUSCLE GROUP", "GYM TYPE", "PHASE", "WEEK"]
)


In [27]:
weekly_df["E1RM_CHANGE"] = (
    weekly_df
    .groupby(["MUSCLE GROUP", "GYM TYPE", "PHASE"])["AVG_E1RM"]
    .diff()
)


In [35]:
weekly_df = weekly_df.sort_values("WEEK")

weekly_df["REL_STRENGTH_CHANGE"] = (
    weekly_df["AVG_RELATIVE_STRENGTH"].diff()
)

In [28]:
weekly_df["STRENGTH_TREND"] = np.where(
    weekly_df["E1RM_CHANGE"] > 0.5,
    "INCREASING",
    np.where(
        weekly_df["E1RM_CHANGE"] < -0.5,
        "DECREASING",
        "STABLE"
    )
)


In [36]:
weekly_df.head(5)

,WEEK,MONTH,MUSCLE GROUP,GYM TYPE,PHASE,AVG_E1RM,AVG_RELATIVE_STRENGTH,TOTAL_VOLUME,E1RM_CHANGE,STRENGTH_TREND,REL_STRENGTH_CHANGE
2,1,Dec,Chest,Soro Gym,Bulk,41.925000,0.590077,2527.5,NaN,STABLE,NaN
5,1,Dec,Triceps,Soro Gym,Bulk,44.958333,0.632770,2017.5,NaN,STABLE,0.042693
12,1,Jan,Triceps,Soro Gym,Bulk,40.722222,0.560912,1630.0,-4.236111,DECREASING,-0.071858
0,1,Dec,Back,Soro Gym,Bulk,52.814815,0.738153,3310.0,NaN,STABLE,0.177240
6,1,Jan,Back,Soro Gym,Bulk,68.166667,0.938935,940.0,15.351852,INCREASING,0.200782


BULK RULE

In [30]:
bulk_issue = (
    (weekly_df["PHASE"] == "BULK") &
    (weekly_df["STRENGTH_TREND"] != "INCREASING")
)

CUT RULE

In [37]:
cut_issue = (
    (weekly_df["PHASE"] == "CUT") &
    (weekly_df["REL_STRENGTH_CHANGE"] < 0)
)

In [38]:
weekly_df["RECOVERY_FLAG"] = np.where(
    bulk_issue | cut_issue,
    "UNDER_RECOVERED",
    "RECOVERED"
)

In [39]:
weekly_df = weekly_df.rename(columns={'MUSCLE GROUP': 'MUSCLE_GROUP', 'GYM TYPE': 'GYM_TYPE'})

In [41]:
weekly_df['RECOVERY_FLAG'].value_counts()

RECOVERY_FLAG
RECOVERED    283
Name: count, dtype: int64

In [42]:
weekly_df

,WEEK,MONTH,MUSCLE_GROUP,GYM_TYPE,PHASE,AVG_E1RM,AVG_RELATIVE_STRENGTH,TOTAL_VOLUME,E1RM_CHANGE,STRENGTH_TREND,REL_STRENGTH_CHANGE,RECOVERY_FLAG
2,1,Dec,Chest,Soro Gym,Bulk,41.925000,0.590077,2527.5,NaN,STABLE,NaN,RECOVERED
5,1,Dec,Triceps,Soro Gym,Bulk,44.958333,0.632770,2017.5,NaN,STABLE,0.042693,RECOVERED
12,1,Jan,Triceps,Soro Gym,Bulk,40.722222,0.560912,1630.0,-4.236111,DECREASING,-0.071858,RECOVERED
0,1,Dec,Back,Soro Gym,Bulk,52.814815,0.738153,3310.0,NaN,STABLE,0.177240,RECOVERED
6,1,Jan,Back,Soro Gym,Bulk,68.166667,0.938935,940.0,15.351852,INCREASING,0.200782,RECOVERED
...,...,...,...,...,...,...,...,...,...,...,...,...
276,52,Dec,Back,Soro Gym,Bulk,52.833333,0.745216,3785.0,-1.076389,DECREASING,0.096628,RECOVERED
277,52,Dec,Biceps,Soro Gym,Bulk,22.089286,0.308925,2152.5,1.324580,INCREASING,-0.436291,RECOVERED
278,52,Dec,Chest,Soro Gym,Bulk,39.208333,0.551843,2715.0,3.388889,INCREASING,0.242918,RECOVERED
281,52,Dec,Shoulders,Soro Gym,Bulk,18.052083,0.253577,1107.5,-0.725694,DECREASING,-0.298266,RECOVERED


In [43]:
weekly_df.to_csv("workout_weekly_analysis.csv", index=False)
